# ViT MNIST — CUDA vs PyTorch — summary notebook

One notebook that runs both implementations end-to-end on a Kaggle 2x T4
session and produces the comparison plots used in the presentation.

Settings: per-rank B=32, Adam lr=1e-3, 3000 steps. Two 1-GPU runs and two
2-GPU runs (so global B=64 for the 2-GPU pair).

Pipeline:
1. Clone repo, install OpenMPI, find NCCL, compile CUDA binary, locate MNIST.
2. **Run 1 — CUDA, 1 GPU** with GPU telemetry sampled in background.
3. **Cooldown** (with plot) — wait for temperature to drop near baseline so the
   second run is not penalised by a hot GPU.
4. **Run 2 — PyTorch, 1 GPU** with telemetry.
5. Cooldown → **Run 3 — CUDA, 2 GPUs** (mpirun) → cooldown → **Run 4 — PyTorch,
   2 GPUs** (torchrun + DDP).
6. Plots:
   - loss/accuracy vs time — **4 curves** (CUDA/PyTorch x 1/2 GPUs).
   - steps/s vs time — 4 curves.
   - throttling (temp + SM clock + power) — **1-GPU pair only** (Runs 1 + 2).

Kaggle setup: GPU T4 x2, Internet On, Digit Recognizer dataset (optional).


## Setup

In [ ]:
!rm -rf /tmp/hpc && git clone https://github.com/SadreevAmir/hpc_final_project /tmp/hpc && cp -r /tmp/hpc/. . && cp src/train_vit_torch.py .
!ls -lh src/train_vit.cu train_vit_torch.py


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv


In [ ]:
import subprocess
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'openmpi-bin', 'libopenmpi-dev'], check=True)
print('OK')


In [ ]:
import os, subprocess

hdr = subprocess.check_output(
    "find /opt/conda /usr/include /usr/local -name nccl.h 2>/dev/null | head -n1",
    shell=True, text=True).strip()
lib = subprocess.check_output(
    "find /opt/conda /usr/lib /usr/local -name \"libnccl.so*\" 2>/dev/null | head -n1",
    shell=True, text=True).strip()
assert hdr, 'nccl.h not found'
assert lib, 'libnccl.so not found'
os.environ['NCCL_INCLUDE'] = os.path.dirname(hdr)
os.environ['NCCL_LIB']     = os.path.dirname(lib)
print('nccl.h     :', hdr)
print('libnccl.so :', lib)


In [ ]:
!mkdir -p bin
!nvcc -O2 -std=c++17 -ccbin mpicxx -arch=sm_75 \
      -I$NCCL_INCLUDE -L$NCCL_LIB \
      -Xlinker -rpath=$NCCL_LIB \
      src/train_vit.cu -o bin/train_vit \
      -lcublas -lnccl
!ls -lh bin/train_vit


In [ ]:
import os, glob, numpy as np

known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
    '/kaggle/input/fashion-mnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(path):
    try:
        with open(path) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed
            if os.path.exists(c) and valid_csv(c)), None)

if CSV is None:
    print('Falling back to torchvision MNIST...')
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr    = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV    = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')

assert os.path.exists(CSV), CSV
print('Using CSV:', CSV)
print('Size     :', os.path.getsize(CSV) // (1024*1024), 'MiB')
os.environ['CSV'] = CSV
!mkdir -p log


## Telemetry helpers

GPU temperature, SM/MEM clocks, power and utilisation are sampled every 250 ms
in a background thread while the training subprocess runs. Telemetry is dumped
to its own CSV with `t_s` offset from the start of the training run, so it
overlays on the training log's `elapsed_s`.

`baseline_temp` is recorded **now** (after compile, before any training), so
the cooldown cell knows what "idle" looks like on this particular T4.

In [ ]:
import os, subprocess, threading, time, csv as csvlib

try:
    import pynvml; pynvml.nvmlInit()
except Exception:
    subprocess.run(["pip", "install", "-q", "nvidia-ml-py"], check=True)
    import pynvml; pynvml.nvmlInit()

gpu0 = pynvml.nvmlDeviceGetHandleByIndex(0)
MAX_SM_MHZ  = pynvml.nvmlDeviceGetMaxClockInfo(gpu0, pynvml.NVML_CLOCK_SM)
MAX_MEM_MHZ = pynvml.nvmlDeviceGetMaxClockInfo(gpu0, pynvml.NVML_CLOCK_MEM)
baseline_temp = pynvml.nvmlDeviceGetTemperature(gpu0,
                                pynvml.NVML_TEMPERATURE_GPU)
print(f"baseline GPU temp: {baseline_temp} C  |  "
      f"max SM clock {MAX_SM_MHZ} MHz  |  max MEM clock {MAX_MEM_MHZ} MHz")


def run_with_telemetry(cmd, telemetry_csv, label="run", env=None):
    """Run cmd (shell str or list) with GPU telemetry sampling in bg.
    Writes telemetry to telemetry_csv. Returns CompletedProcess."""
    samples, stop = [], threading.Event()

    def monitor():
        t0 = time.time()
        while not stop.is_set():
            try:
                temp = pynvml.nvmlDeviceGetTemperature(gpu0,
                            pynvml.NVML_TEMPERATURE_GPU)
                sm   = pynvml.nvmlDeviceGetClockInfo(gpu0,
                            pynvml.NVML_CLOCK_SM)
                mem  = pynvml.nvmlDeviceGetClockInfo(gpu0,
                            pynvml.NVML_CLOCK_MEM)
                pwr  = pynvml.nvmlDeviceGetPowerUsage(gpu0) / 1000.0
                util = pynvml.nvmlDeviceGetUtilizationRates(gpu0).gpu
                samples.append((time.time() - t0, temp, sm, mem, pwr, util))
            except Exception:
                pass
            time.sleep(0.25)

    th = threading.Thread(target=monitor, daemon=True)
    th.start()
    t_proc = time.time()
    res = subprocess.run(cmd, shell=isinstance(cmd, str),
                         capture_output=True, text=True, env=env)
    elapsed = time.time() - t_proc
    stop.set(); th.join(timeout=2)

    os.makedirs(os.path.dirname(telemetry_csv) or ".", exist_ok=True)
    with open(telemetry_csv, "w", newline="") as f:
        w = csvlib.writer(f)
        w.writerow(["t_s", "temp_c", "sm_mhz", "mem_mhz",
                    "power_w", "gpu_util"])
        w.writerows(samples)

    tail = "\n".join(res.stdout.strip().splitlines()[-12:])
    print(f"[{label}] finished in {elapsed:.1f}s, "
          f"{len(samples)} telemetry samples -> {telemetry_csv}")
    print("--- last 12 stdout lines ---")
    print(tail)
    if res.returncode != 0:
        print("STDERR (last 500 chars):")
        print(res.stderr[-500:])
    return res


def cooldown(delta=5, max_wait=300, sample_s=2.0, print_every=10.0):
    """Sample GPU temp every sample_s seconds until temp <= baseline+delta
    or max_wait elapses. Writes log/cooldown.csv and returns the samples."""
    target = baseline_temp + delta
    samples, t0, next_print = [], time.time(), 0.0
    reason = None
    while True:
        temp = pynvml.nvmlDeviceGetTemperature(gpu0,
                    pynvml.NVML_TEMPERATURE_GPU)
        el = time.time() - t0
        samples.append((el, temp))
        if el >= next_print:
            print(f"  {el:5.0f}s  temp={temp} C  (target {target})",
                  flush=True)
            next_print += print_every
        if temp <= target:
            reason = f"cooled to {temp} C after {el:.0f}s"; break
        if el >= max_wait:
            reason = f"cap {max_wait}s hit at {temp} C"; break
        time.sleep(sample_s)
    print(reason, f"(baseline {baseline_temp} + delta {delta})")
    os.makedirs("log", exist_ok=True)
    with open("log/cooldown.csv", "w", newline="") as f:
        w = csvlib.writer(f); w.writerow(["t_s", "temp_c"])
        w.writerows(samples)
    return samples


## Run 1 — CUDA 1 GPU (3000 steps)

`./bin/train_vit` writes `training_log.csv`; we rename it to
`log/cuda_1gpu_log.csv`. All perf flags default to ON (`all_fast` config:
batched attention, sgemv bias, fast LN bwd, narrow memset).

In [ ]:
cuda_env = {**os.environ,
            "PERF_LN_BWD_FAST":   "0",   # shelved fix
            "PERF_SGEMV_BIAS":    "1",
            "PERF_ATTN_BATCHED":  "1",
            "PERF_NARROW_MEMSET": "1"}

res = run_with_telemetry(
    ['./bin/train_vit', os.environ['CSV'], '3000', '32', '0.001'],
    'log/cuda_telemetry.csv',
    label='CUDA 1GPU',
    env=cuda_env)
os.replace('training_log.csv', 'log/cuda_1gpu_log.csv')
print('saved: log/cuda_1gpu_log.csv')


## Cooldown

Wait until GPU temperature drops back near the baseline (recorded above).
Default: baseline + 5 °C, cap at 5 min. Without this the PyTorch run starts
on a hot GPU and its SM clock starts throttled.

In [ ]:
cooldown(delta=5, max_wait=300)


## Cooldown curve

Small plot of temperature vs time during the cooldown above.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_cd = pd.read_csv('log/cooldown.csv')
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_cd['t_s'], df_cd['temp_c'], color='tab:purple',
        linewidth=2, marker='o', markersize=3)
ax.axhline(baseline_temp, color='k', linestyle=':', linewidth=1, alpha=0.7,
           label=f'baseline {baseline_temp} C')
ax.axhline(baseline_temp + 5, color='k', linestyle='--', linewidth=1, alpha=0.5,
           label=f'target {baseline_temp + 5} C')
ax.set_xlabel('Time since cooldown start (s)')
ax.set_ylabel('GPU temperature (C)')
start_t, end_t = df_cd["temp_c"].iloc[0], df_cd["temp_c"].iloc[-1]
ax.set_title(f'Cooldown between runs: {start_t} C -> {end_t} C in '
             f'{df_cd["t_s"].iloc[-1]:.0f}s')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('summary_cooldown.png', dpi=130, bbox_inches='tight')
plt.show()


## Run 2 — PyTorch 1 GPU (3000 steps)

Same B=32, lr=1e-3, 3000 steps. PyTorch attention pinned to
`SDPBackend.MATH` inside the script so we compare like-for-like with the
naive CUDA implementation. Log written via `--log-path`.

In [ ]:
res = run_with_telemetry(
    ['python', 'train_vit_torch.py', os.environ['CSV'],
     '3000', '32', '0.001', '--device', 'cuda',
     '--log-path', 'log/torch_1gpu_log.csv'],
    'log/torch_telemetry.csv',
    label='PyTorch 1GPU')
print('saved: log/torch_1gpu_log.csv')


## Cooldown (before CUDA 2-GPU run)


In [ ]:
cooldown(delta=5, max_wait=300)


## Run 3 — CUDA, 2 GPUs (3000 steps)

Per-rank B=32 → global B=64. `mpirun -np 2` spawns one rank per GPU; gradients are NCCL all-reduced. Telemetry is recorded for completeness but the throttling plot below only uses the 1-GPU pair.


In [ ]:
res = run_with_telemetry(
    ['mpirun', '--allow-run-as-root', '-np', '2',
     './bin/train_vit', os.environ['CSV'],
     '3000', '32', '0.001'],
    'log/cuda_2gpu_telemetry.csv',
    label='CUDA 2GPU',
    env=cuda_env)
os.replace('training_log.csv', 'log/cuda_2gpu_log.csv')
print('saved: log/cuda_2gpu_log.csv')


## Cooldown (before PyTorch 2-GPU run)


In [ ]:
cooldown(delta=5, max_wait=300)


## Run 4 — PyTorch, 2 GPUs (3000 steps)

`torchrun --nproc_per_node=2` → one rank per GPU; DDP averages gradients during backward. Per-rank B=32 → global B=64, same lr/steps as the other runs.


In [ ]:
res = run_with_telemetry(
    ['torchrun', '--standalone', '--nproc_per_node=2',
     '--master_port=29541', 'train_vit_torch.py',
     os.environ['CSV'], '3000', '32', '0.001',
     '--device', 'cuda',
     '--log-path', 'log/torch_2gpu_log.csv'],
    'log/torch_2gpu_telemetry.csv',
    label='PyTorch 2GPU')
print('saved: log/torch_2gpu_log.csv')


## Plot 1 — loss and accuracy vs wall-clock time (4 curves)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

runs = [
    ('CUDA 1 GPU',     'log/cuda_1gpu_log.csv',  'tab:blue', '-'),
    ('CUDA 2 GPUs',    'log/cuda_2gpu_log.csv',  'tab:blue', '--'),
    ('PyTorch 1 GPU',  'log/torch_1gpu_log.csv', 'tab:red',  '-'),
    ('PyTorch 2 GPUs', 'log/torch_2gpu_log.csv', 'tab:red',  '--'),
]

fig, (ax_l, ax_a) = plt.subplots(1, 2, figsize=(14, 5))
for label, path, color, ls in runs:
    df = pd.read_csv(path)
    sm_l = df['loss'].rolling(10, min_periods=1).mean()
    sm_a = df['accuracy'].rolling(10, min_periods=1).mean()
    ax_l.plot(df['elapsed_s'], sm_l, color=color, linestyle=ls,
              linewidth=2, label=label)
    ax_a.plot(df['elapsed_s'], sm_a, color=color, linestyle=ls,
              linewidth=2, label=label)

ax_l.set_xlabel('Wall-clock time, s'); ax_l.set_ylabel('Cross-entropy loss')
ax_l.set_title('Loss vs time'); ax_l.legend(); ax_l.grid(alpha=0.3)
ax_a.set_xlabel('Wall-clock time, s'); ax_a.set_ylabel('Accuracy')
ax_a.set_title('Accuracy vs time'); ax_a.legend(); ax_a.grid(alpha=0.3)
ax_a.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
plt.suptitle('CUDA vs PyTorch x 1/2 GPUs - Adam lr=1e-3, per-rank B=32, 3000 steps',
             fontsize=13)
plt.tight_layout()
plt.savefig('summary_loss_acc.png', dpi=150, bbox_inches='tight')
plt.show()


## Plot 2 — steps per second vs time (4 curves)


In [ ]:
runs = [
    ('CUDA 1 GPU',     'log/cuda_1gpu_log.csv',  'tab:blue', '-'),
    ('CUDA 2 GPUs',    'log/cuda_2gpu_log.csv',  'tab:blue', '--'),
    ('PyTorch 1 GPU',  'log/torch_1gpu_log.csv', 'tab:red',  '-'),
    ('PyTorch 2 GPUs', 'log/torch_2gpu_log.csv', 'tab:red',  '--'),
]

fig, ax = plt.subplots(figsize=(11, 5))

for label, path, color, ls in runs:
    df = pd.read_csv(path).sort_values('step').reset_index(drop=True)
    d_step    = df['step'].diff()
    d_elapsed = df['elapsed_s'].diff()
    sps       = d_step / d_elapsed
    t_mid     = (df['elapsed_s'] + df['elapsed_s'].shift()) / 2
    valid = d_elapsed > 0
    t_mid, sps = t_mid[valid], sps[valid]

    ax.plot(t_mid, sps.rolling(window=10, min_periods=1).mean(),
            color=color, linestyle=ls, linewidth=2, label=label)

    overall = df['step'].iloc[-1] / df['elapsed_s'].iloc[-1]
    print(f'{label:<16} mean {sps.mean():6.1f} steps/s  |  '
          f'overall {overall:6.1f} steps/s  '
          f'({df["elapsed_s"].iloc[-1]:.1f} s total)')

ax.set_xlabel('Wall-clock time, s')
ax.set_ylabel('Training speed, steps / second')
ax.set_title('Steps per second vs time - CUDA vs PyTorch x 1/2 GPUs')
ax.grid(alpha=0.3); ax.set_ylim(bottom=0); ax.legend()
plt.tight_layout()
plt.savefig('summary_steps_per_sec.png', dpi=150, bbox_inches='tight')
plt.show()


## Plot 3 — throttling: temperature, SM clock, power (1-GPU pair only)

Telemetry sampled at 250 ms during each run. The 2-GPU runs are not shown
here — they were run with a hot/cold history different from the 1-GPU pair,
and the throttling story we care about is the apples-to-apples
CUDA-1GPU vs PyTorch-1GPU comparison (Run 1 + Run 2).

SM clock dropping below the dashed `max` line means thermal throttle.
Temperature panel shows the cooldown effect: the second run starts cold.


In [ ]:
tel = [(lbl, pd.read_csv(p), col) for lbl, p, col in [
    ('CUDA 1 GPU',    'log/cuda_telemetry.csv',  'tab:blue'),
    ('PyTorch 1 GPU', 'log/torch_telemetry.csv', 'tab:red'),
]]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
ax_t, ax_c, ax_p = axes

for lbl, df, color in tel:
    ax_t.plot(df['t_s'], df['temp_c'],  color=color, linewidth=1.5, label=lbl)
    ax_c.plot(df['t_s'], df['sm_mhz'],  color=color, linewidth=1.5, label=lbl)
    ax_p.plot(df['t_s'], df['power_w'], color=color, linewidth=1.5, label=lbl)

ax_t.axhline(baseline_temp, color='k', linestyle=':', linewidth=1,
             alpha=0.6, label=f'baseline {baseline_temp} C')
ax_t.set_ylabel('Temperature (C)'); ax_t.set_title('GPU temperature')
ax_t.legend(); ax_t.grid(alpha=0.3)

ax_c.axhline(MAX_SM_MHZ, color='k', linestyle='--', linewidth=1,
             alpha=0.6, label=f'max {MAX_SM_MHZ} MHz')
ax_c.set_ylabel('SM clock (MHz)'); ax_c.set_title('SM clock — dips = throttle')
ax_c.legend(); ax_c.grid(alpha=0.3); ax_c.set_ylim(bottom=0)

ax_p.set_ylabel('Power (W)'); ax_p.set_xlabel('Time since run start (s)')
ax_p.set_title('Power draw'); ax_p.legend(); ax_p.grid(alpha=0.3)
ax_p.set_ylim(bottom=0)

plt.suptitle('Throttling view — telemetry sampled at 250 ms',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('summary_throttling.png', dpi=150, bbox_inches='tight')
plt.show()

# Quick summary table
print()
print(f'{"run":<16}{"temp peak":>12}{"sm avg":>12}{"sm min":>12}'
      f'{"power avg":>12}')
for lbl, df, _ in tel:
    print(f'{lbl:<16}{df["temp_c"].max():>10.0f} C'
          f'{df["sm_mhz"].mean():>9.0f} MHz'
          f'{df["sm_mhz"].min():>9.0f} MHz'
          f'{df["power_w"].mean():>9.1f} W')
